## Data Cleaning Markdown
- Collect data from Hugging Face
- Subset for variables of interest
- Plot the variables of interest in different ways
- Inspect the data
- Check for NAs
- Clean

### PHASE 1: Data Cleaning

In [2]:
# Setup
from datasets import load_dataset
import numpy as np
import pandas as pd
import re

/Users/mattie/Desktop/Masters/Data Science/DataScience2026/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load data set for only the food products
ds = load_dataset("openfoodfacts/product-database", split="food")

In [22]:
# columns of interest
columns_of_interest = [
    "product_name",
    "brands",
    "nova_group",
    "nutriscore_score",
    "nutriscore_grade",
    "lang",
    "ingredients_original_tags",
    "code"
]

# convert to df
df = ds.select_columns(columns_of_interest).to_pandas()

In [24]:
# save to csv for future use
df.to_csv("data/raw_data.csv", index=False)

In [25]:
# Inspect first few rows of the dataset
df.head()

,product_name,brands,nova_group,nutriscore_score,nutriscore_grade,lang,ingredients_original_tags,code
0,"[{'lang': 'main', 'text': 'Véritable pâte à ta...",Bovetti,NaN,25.0,e,fr,None,0000101209159
1,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...",Lagg's,1.0,NaN,unknown,en,[en:camomile-flower],0000105000011
2,"[{'lang': 'main', 'text': 'Lagg's, herbal tea,...",Lagg's,1.0,NaN,unknown,en,[en:peppermint],0000105000042
3,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...",Lagg's,NaN,NaN,unknown,en,[en:linden-flowers],0000105000059
4,"[{'lang': 'main', 'text': 'Herbal Tea, Hibiscu...",Lagg's,NaN,NaN,unknown,en,[en:roselle-flower],0000105000073


In [26]:
# See rows and columns
df.shape

(4401229, 8)

In [27]:
# filter for only rows where lang is 'en'
df = df[df["lang"] == "en"]

df.shape

(1816432, 8)

In [28]:
# Build a function to clean columns removing non-alphabetical characters
def lower_remove_nonalpha(name: str) -> str:
    if not isinstance(name, str):
        return name  # or return "" / raise — your call
    return re.sub(r'[^a-z]', '', name.lower())

df["product_name_clean"] = df["product_name"].apply(lower_remove_nonalpha)

df.shape

(1816432, 9)

In [39]:
# concatenate brand and product name to create a unique product ID - if brand is na then just use product name
df["product_ID"] = np.where(
    df["brands"].isna(),
    df["product_name_clean"].astype(str),
    df["brands"] + "_" + df["product_name_clean"].astype(str)
)

# make sure no products duplicated by "code" (barcode) - also unique product ID
if df["code"].nunique() != len(df):
    print("Warning: code column has duplicates.")

# for code duplicates keep the instance with the most data (least missing values)
df["missing_values"] = df.isna().sum(axis=1)
df = df.sort_values("missing_values").drop_duplicates(subset="code", keep="first").drop(columns="missing_values")

# double check for duplicates after dropping
if df["code"].nunique() != len(df):
    print("Warning: code column has duplicates.")
else:
    print("No duplicates in code column.")

No duplicates in code column.


In [40]:
df.shape

(1816426, 10)

### PHASE 2: Data Inspection & Plotting

In [41]:
# count na in each column
na_count = df.isna().sum()

# get percentage of missing values in each column
na_percent = df.isna().mean()*100

# combine into a dataframe
na_df = pd.DataFrame({"na_count": na_count, "na_percent": na_percent})
print(na_df)

                           na_count  na_percent
product_name                      0    0.000000
brands                       623861   34.345522
nova_group                  1293367   71.203947
nutriscore_score            1306173   71.908957
nutriscore_grade              44046    2.424872
lang                              0    0.000000
ingredients_original_tags   1193950   65.730726
code                              0    0.000000
product_name_clean                0    0.000000
product_ID                        0    0.000000


OBS: We have quite a lot of data without a nova group or a numerical nutriscore score. But many more do have a nutrigrade. So we should probably use the grade instead of the score - I will continue by using the nutrigrade.

Quite a lot of the data also does not have ingredients original tags - perhaps we need to check how much we have and if we use a different column from the OG data.

#### Create filtered df where there are ingredients and nutriscores

In [47]:
# filter for rows with a nutriscore_grade and ingredients_original_tags
df_filtered = df[df["nutriscore_grade"].notna() & df["ingredients_original_tags"].notna()]

df_filtered.shape

(621135, 10)

#### Create df with novas

In [56]:
# create df where nova is present
with_nova = df_filtered[df_filtered["nova_group"].notna()]

# save to csv for modelling
with_nova.to_csv("data/with_nova.csv", index=False)

with_nova.shape

(514009, 10)

In [65]:
# see how many products fall into each nova score for the with nova df
print(with_nova["nova_group"].value_counts())

nova_group
4.0    367187
3.0     87634
1.0     50922
2.0      8266
Name: count, dtype: int64


In [66]:
# see how many products fall into each nutriscore grade for the missing nova df
print(with_nova["nutriscore_grade"].value_counts())

nutriscore_grade
e                 121384
unknown           115197
d                  94224
c                  75832
a                  57453
b                  38103
not-applicable     11816
Name: count, dtype: int64


#### Create df without novas

In [55]:
# see how many rows of the filtered dataset have a nova group
missing_nova = df_filtered[df_filtered["nova_group"].isna()]

# save to csv for Nova imputation
missing_nova.to_csv("data/missing_nova.csv", index=False)

missing_nova.shape

(107126, 10)

In [62]:
# percentage of df_filtered that doesn't have a nova
percent_missing_nova = (missing_nova.shape[0] / df_filtered.shape[0]) * 100
print(f"Percentage of df_filtered without nova_group: {percent_missing_nova:.2f}%")

Percentage of df_filtered without nova_group: 17.25%


In [ ]:
# see how many products fall into each nutriscore grade for the missing nova df
print(missing_nova["nutriscore_grade"].value_counts())

Issues to be addressed?
- For the nutriscore grade there is some called unknown and not-applicable -> need to see what we want to do with these? Filter out? or see what the use cases are?
- There are varying amount in each nova group (when we split the data again it should be equal between all groups in each df? Amount of data could be seen as a weight?)